# PTDR End-to-End OCR Inference

This notebook loads one DBNet checkpoint and one PARSeq checkpoint, runs end-to-end OCR on a PTDR image, and visualizes the intermediate steps.

Metrics used by the shared evaluator:
- End-to-end precision / recall: a predicted polygon-text pair is correct when polygon IoU is at least `MATCH_IOU_THR` and the normalized transcript matches exactly.
- Keyword recall: per-image unique ground-truth whitespace-delimited tokens that appear anywhere in the recognized output for that image.

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
from IPython.display import JSON, display

REPO_ROOT = Path.cwd().resolve()
repo_root_hint = __import__('os').environ.get('PTDR_REPO_ROOT')
candidate_roots = [REPO_ROOT, *REPO_ROOT.parents]
if repo_root_hint:
    candidate_roots.append(Path(repo_root_hint).expanduser())
for candidate in candidate_roots:
    if (candidate / 'experiments' / 'ptdr').exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError('Could not locate the PTDR repo root from the current notebook working directory.')
sys.path.insert(0, str(REPO_ROOT / 'experiments' / 'ptdr'))

from end_to_end_utils import (
    evaluate_records,
    load_dbnet_detector,
    load_detection_records,
    load_parseq_recognizer,
    render_detection_overlay,
    render_recognition_gallery,
    run_end_to_end_inference,
)


In [ ]:
def latest_existing(paths):
    candidates = [Path(path) for path in paths]
    existing = [path for path in candidates if path.exists()]
    if not existing:
        raise FileNotFoundError(f'None of these checkpoint candidates exist: {candidates}')
    return max(existing, key=lambda path: path.stat().st_mtime)

DBNET_CANDIDATES = [
    REPO_ROOT / 'work_dirs' / 'dbnet_r50_ptdr_scene' / 'epoch_4.pth',
    REPO_ROOT / 'work_dirs' / 'dbnet_r18_ptdr_scene' / 'epoch_4.pth',
    REPO_ROOT / 'work_dirs' / 'dbnet_r50_ptdr_scene' / 'epoch_3.pth',
    REPO_ROOT / 'work_dirs' / 'dbnet_r18_ptdr_scene' / 'epoch_3.pth',
]
PARSEQ_CANDIDATES = [
    REPO_ROOT / 'work_dirs' / 'parseq_default_ptdr_scene' / 'checkpoints' / 'last.ckpt',
    REPO_ROOT / 'work_dirs' / 'parseq_tiny_ptdr_scene' / 'checkpoints' / 'last.ckpt',
    REPO_ROOT / 'work_dirs' / 'parseq_ptdr_scene' / 'checkpoints' / 'last.ckpt',
]

DBNET_CHECKPOINT = latest_existing(DBNET_CANDIDATES)
PARSEQ_CHECKPOINT = latest_existing(PARSEQ_CANDIDATES)
SPLIT = 'val'
DEVICE = 'cuda:0' if __import__('torch').cuda.is_available() else 'cpu'
DET_SCORE_THR = 0.3
MATCH_IOU_THR = 0.5
RECOGNITION_BATCH_SIZE = 32
IMAGE_INDEX = None  # set an integer for a deterministic image selection

display(JSON({
    'dbnet_checkpoint': str(DBNET_CHECKPOINT),
    'parseq_checkpoint': str(PARSEQ_CHECKPOINT),
    'device': DEVICE,
}))


In [ ]:
detector = load_dbnet_detector(DBNET_CHECKPOINT, repo_root=REPO_ROOT, device=DEVICE)
recognizer = load_parseq_recognizer(PARSEQ_CHECKPOINT, device=DEVICE)
manifest_path = detector.manifest_paths[SPLIT]
records = load_detection_records(manifest_path)
display(JSON({'manifest_path': str(manifest_path), 'image_count': len(records)}))


In [ ]:
sample_record = random.choice(records) if IMAGE_INDEX is None else records[IMAGE_INDEX]
result, image_rgb, crops = run_end_to_end_inference(
    record=sample_record,
    detector=detector,
    recognizer=recognizer,
    repo_root=REPO_ROOT,
    split=SPLIT,
    det_score_thr=DET_SCORE_THR,
    match_iou_thr=MATCH_IOU_THR,
    recognition_batch_size=RECOGNITION_BATCH_SIZE,
)
display(JSON({
    'image_path': result.image_path,
    'end_to_end_precision': result.end_to_end_precision,
    'end_to_end_recall': result.end_to_end_recall,
    'keyword_recall': result.keyword_recall,
    'true_positives': result.true_positives,
    'gt_count': result.gt_count,
    'pred_count': result.pred_count,
    'keyword_hits': result.keyword_hits,
    'keyword_total': result.keyword_total,
    'gt_keywords': result.gt_keywords,
    'pred_keywords': result.pred_keywords,
}))


In [ ]:
display(JSON({
    'ground_truth': [
        {'index': index, 'text': word.text, 'normalized_text': word.normalized_text}
        for index, word in enumerate(result.ground_truth)
    ],
    'predictions_top_20': [
        {
            'index': index,
            'score': word.score,
            'text': word.text,
            'normalized_text': word.normalized_text,
        }
        for index, word in sorted(enumerate(result.predictions), key=lambda item: item[1].score, reverse=True)[:20]
    ],
    'matches': [
        {'gt_index': match.gt_index, 'pred_index': match.pred_index, 'iou': match.iou}
        for match in result.matches
    ],
}))


In [ ]:
overlay = render_detection_overlay(
    image_rgb=image_rgb,
    ground_truth=result.ground_truth,
    predictions=result.predictions,
    matches=result.matches,
)
gallery = render_recognition_gallery(crops=crops, predictions=result.predictions)

fig, axes = plt.subplots(3, 1, figsize=(18, 24))
axes[0].imshow(image_rgb)
axes[0].set_title('Original image')
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title('Ground truth vs predicted polygons and transcripts')
axes[1].axis('off')

axes[2].imshow(gallery)
axes[2].set_title('Warped detection crops sent to PARSeq')
axes[2].axis('off')

plt.tight_layout()


In [ ]:
# Smoke-test the evaluator on a small subset first. Remove limit=None to run the whole split.
summary, _results = evaluate_records(
    records=records,
    detector=detector,
    recognizer=recognizer,
    repo_root=REPO_ROOT,
    split=SPLIT,
    det_score_thr=DET_SCORE_THR,
    match_iou_thr=MATCH_IOU_THR,
    recognition_batch_size=RECOGNITION_BATCH_SIZE,
    limit=10,
)
display(JSON(summary))
